In [ ]:
!pip install rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 28.9 MB/s eta 0:00:00


In [ ]:
# Cell: Load, Process Multiple Hotspots, and Save

import rasterio
import numpy as np
import math

# --- Your cloudburst parameters ---
center_points = [
    # (76.13, 11.68),
    (75.97418981928425, 11.798547794903879),
    (76.11289221186237, 11.60355874564377),
    (76.1746903075655, 11.763594370588798)
]
max_rainfall = 150
radius_m = 7000

# --- File paths ---
input_tif = '/content/drive/MyDrive/Wayanad_Exports/Cloud Burst rain/avg_rainfall_for_CloudBurstProcessingIncolab2025.tif'
output_tif = '/content/drive/MyDrive/Wayanad_Exports/Cloud Burst rain/2025mean_rainfall_100m_CloudBurst_3Points.tif'

# Open the source GeoTIFF
with rasterio.open(input_tif) as src:
    profile = src.profile
    rainfall_array = src.read(1)
    transform = src.transform
    print("--- Original File Metadata (Profile) ---")
    print(profile)
    print("----------------------------------------")

# --- PIXEL-LEVEL PROCESSING (IMPROVED METHOD) ---

# 1. Create a separate array to hold the cloudburst effect, initialized to zeros.
cloudburst_effect_layer = np.zeros_like(rainfall_array, dtype='float32')

rows, cols = rainfall_array.shape
pixel_width_deg = transform[0]
radius_in_pixels = radius_m / (pixel_width_deg * 111320)

# 2. Iterate through each pixel ONCE
for r in range(rows):
    for c in range(cols):
        max_rain_to_add_at_this_pixel = 0

        # For the current pixel (r, c), check its distance to EACH center point
        for center_lon, center_lat in center_points:
            center_col, center_row = ~transform * (center_lon, center_lat)
            center_col, center_row = int(center_col), int(center_row)

            distance = math.sqrt((r - center_row)**2 + (c - center_col)**2)

            # If it's within the radius of the current center point...
            if distance < radius_in_pixels:
                # Calculate the potential rainfall to add from this center
                potential_rain = max_rainfall * (1 - (distance / radius_in_pixels))
                # Keep track of the HIGHEST potential rainfall for this pixel
                if potential_rain > max_rain_to_add_at_this_pixel:
                    max_rain_to_add_at_this_pixel = potential_rain

        # After checking all centers, assign the max value to the effects layer
        cloudburst_effect_layer[r, c] = max_rain_to_add_at_this_pixel

# 3. After the loop, add the entire effects layer to the base rainfall array in one operation
final_rainfall_array = rainfall_array.astype('float32') + cloudburst_effect_layer

print("\nCloudburst simulation complete.")

# --- SAVE THE NEW FILE ---
profile.update(dtype='float32')
with rasterio.open(output_tif, 'w', **profile) as dst:
    dst.write(final_rainfall_array, 1)

print(f"New image saved successfully to: {output_tif}")

In [ ]:
# Cell: Load, Process Multiple INDIVIDUAL Hotspots, and Save

import rasterio
import numpy as np
import math

# --- Your cloudburst parameters (NEW STRUCTURE) ---
# Each dictionary represents a unique cloudburst event.
hotspots = [
    # Group 1: An intense, focused storm (150mm rain, 7km radius)
    # {'lon': 75.97418981928425, 'lat': 11.798547794903879, 'rain': 150, 'radius_m': 7000},
    # {'lon': 76.11289221186237, 'lat': 11.60355874564377, 'rain': 150, 'radius_m': 7000},
    # {'lon': 76.1746903075655, 'lat': 11.763594370588798, 'rain': 150, 'radius_m': 7000},

    # Group 2: A wider, less intense system (150mm rain, different radius)
    # {'lon': 76.19254309076862, 'lat': 11.554184660848017, 'rain': 150, 'radius_m': 6000},
    # {'lon': 76.00714880365925, 'lat': 11.585397621034303, 'rain': 150, 'radius_m': 6000},
    # {'lon': 76.092292846628, 'lat': 11.688967410848479, 'rain': 150, 'radius_m': 5000},
    # {'lon': 75.95633703608112, 'lat': 11.731729135069404, 'rain': 150, 'radius_m': 5000},
    # {'lon': 75.97281652826862, 'lat': 11.82449113598898, 'rain': 150, 'radius_m': 4000},
    # {'lon': 76.04697424311237, 'lat': 11.900425010826122, 'rain': 150, 'radius_m': 3000},

    # # Group 2: A wider, less intense system (150mm rain, different radius)
    # {'lon': 75.97418981928425, 'lat': 11.798547794903879, 'rain': 150, 'radius_m': 6000},
    # {'lon': 76.11289221186237, 'lat': 11.60355874564377, 'rain': 150, 'radius_m': 6000},
    # {'lon': 75.850593627878, 'lat': 11.786852361170737, 'rain': 150, 'radius_m': 5000},
    # {'lon': 76.00714880365925, 'lat': 11.677939761793779, 'rain': 150, 'radius_m': 5000},

    # Group 2: A wider, less intense system (150mm rain, different radius)
    {'lon': 75.97418981928425, 'lat': 11.798547794903879, 'rain': 150, 'radius_m': 7000},
    {'lon': 76.11289221186237, 'lat': 11.60355874564377, 'rain': 150, 'radius_m': 7000},
    {'lon': 76.03049475092487, 'lat': 11.567638960443373, 'rain': 150, 'radius_m': 7000},
    {'lon': 75.95771032709675, 'lat': 11.700801523621138, 'rain': 150, 'radius_m': 7000},

]

# --- File paths ---
input_tif = '/content/drive/MyDrive/Wayanad_Exports/Cloud Burst rain/avg_rainfall_for_CloudBurstProcessingIncolab2025.tif'
output_tif = '/content/drive/MyDrive/Wayanad_Exports/Cloud Burst rain/2025mean_rainfall_100m_CloudBurst_4Points.tif'

# Open the source GeoTIFF
with rasterio.open(input_tif) as src:
    profile = src.profile
    rainfall_array = src.read(1)
    transform = src.transform
    print("--- Original File Metadata (Profile) ---")
    print(profile)
    print("----------------------------------------")

# --- PIXEL-LEVEL PROCESSING (MODIFIED METHOD) ---

# 1. Create a separate array to hold the cloudburst effect, initialized to zeros.
cloudburst_effect_layer = np.zeros_like(rainfall_array, dtype='float32')

rows, cols = rainfall_array.shape
pixel_width_deg = transform[0]

# 2. Pre-process the hotspots data for efficiency
# We calculate pixel coordinates and radius in pixels just once before the main loop.
for spot in hotspots:
    spot['col'], spot['row'] = ~transform * (spot['lon'], spot['lat'])
    spot['radius_px'] = spot['radius_m'] / (pixel_width_deg * 111320)

# 3. Iterate through each pixel ONCE
for r in range(rows):
    for c in range(cols):
        max_rain_to_add_at_this_pixel = 0

        # For the current pixel (r, c), check its distance to EACH hotspot
        for spot in hotspots:
            center_row, center_col = int(spot['row']), int(spot['col'])

            distance = math.sqrt((r - center_row)**2 + (c - center_col)**2)

            # Check if pixel is within this specific hotspot's radius
            if distance < spot['radius_px']:
                # Calculate potential rain using this hotspot's specific parameters
                potential_rain = spot['rain'] * (1 - (distance / spot['radius_px']))

                # Keep track of the HIGHEST potential rainfall from any hotspot
                if potential_rain > max_rain_to_add_at_this_pixel:
                    max_rain_to_add_at_this_pixel = potential_rain

        # After checking all hotspots, assign the max value to the effects layer
        cloudburst_effect_layer[r, c] = max_rain_to_add_at_this_pixel

# 4. After the loop, add the entire effects layer to the base rainfall array
final_rainfall_array = rainfall_array.astype('float32') + cloudburst_effect_layer

print("\nCloudburst simulation complete.")

# --- SAVE THE NEW FILE ---
profile.update(dtype='float32')
with rasterio.open(output_tif, 'w', **profile) as dst:
    dst.write(final_rainfall_array, 1)

print(f"New image saved successfully to: {output_tif}")

--- Original File Metadata (Profile) ---
{'driver': 'GTiff', 'dtype': 'float64', 'nodata': None, 'width': 749, 'height': 588, 'count': 1, 'crs': CRS.from_wkt('GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]'), 'transform': Affine(0.0008983152841195215, 0.0, 75.77199590019752,
       0.0, -0.0008983152841195215, 11.97903431373382), 'blockxsize': 256, 'blockysize': 256, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
----------------------------------------

Cloudburst simulation complete.
New image saved successfully to: /content/drive/MyDrive/Wayanad_Exports/Cloud Burst rain/2025mean_rainfall_100m_CloudBurst_4Points.tif
